# Dense X Retrieval — Downscaled Replication (SEG301)

**Paper:** Chen et al., *Dense X Retrieval: What Retrieval Granularity Should We Use?* (EMNLP 2024)
**Repo tác giả:** `chentong0/factoid-wiki` — chỉ có data (FactoidWiki) + Propositionizer, **không có code evaluation** → phần eval viết lại theo Sec 4.3, 5 và Appendix của paper.
**Mục tiêu:** kiểm chứng **xu hướng tương đối** Proposition > Sentence > Passage, không khớp số tuyệt đối.

## 1. Plan replicate

| Hạng mục | Paper | Replicate (notebook này) |
|---|---|---|
| Corpus | Wikipedia 2021-10-13 (FactoidWiki), 6M pages | **8,000 pages** lấy nguyên từ FactoidWiki = gold pages + distractor ngẫu nhiên (≠ gold) |
| Granularity | passage (41M) / sentence (114M) / proposition (257M) | cùng 3 loại, đúng unit của tác giả (không tự chạy Propositionizer) |
| Dataset | NQ, TQA, WebQ, SQuAD, EQ (full test) | **SQuAD** v1.1 dev + **NQ-open test**, mỗi dataset **500 câu** random (seed 42) |
| Gold title | — | SQuAD: field `title` · NQ: join câu hỏi với official NQ dev → `document.title` |
| Chống leak | — | test split như paper; loại câu trùng train; demo 4-shot lấy từ train; distractor ∩ gold = ∅ |
| Retriever | SimCSE, Contriever (unsup.) · DPR, GTR (sup.) | **giữ nguyên 4 retriever**, đúng checkpoint HF trong Appendix |
| Map unit → passage | max score unit trong passage | giống paper |
| Retrieval metric | Passage Recall@5/20 (Table 3), Recall@l words (Fig 4/7) | giống paper + Recall@1…100, MRR, gold-page recall |
| QA reader 1 | FiD T5-large `nq_reader_large`, top-5/20 passages (Table 4) | **giống paper** (checkpoint gốc, nạp vào transformers mới) |
| QA reader 2 | LLaMA-2-7B 4-shot, EM@100/500 tokens (Table 5) | giống paper (vLLM, greedy) + baseline closed-book |
| Phân tích khác | Fig 3 (entity frequency, EQ), Table 6/7 (lỗi) | Fig 3 chuyển thể (độ hiếm answer), Table 6/7, bootstrap CI, McNemar, theo loại câu hỏi, chi phí index |
| Không làm | — | Table 2/Figure 5 (train Propositionizer); TQA/WebQ/EQ |

## 2. Các stage (bật bằng `PRESET` / `RUN` ở cell đầu tiên)

| Stage | Làm gì | Máy | Thời gian ước tính | Output chính |
|---|---|---|---|---|
| `data` | tải câu hỏi, check leak, verify title, dựng corpus 8,000 pages | CPU hoặc GPU, Internet ON | 1.5–3h | `corpus/`, `questions.jsonl`, `demos.jsonl`, `corpus_meta.json` |
| `retrieval` | encode + search 4 retriever × 3 granularity | **GPU T4×2** | 1–2h | `ranked/*.npz`, `results/efficiency.csv` |
| `fid` | Table 4 — FiD trên top-5/20 passages | **GPU T4×2** | 1–2h | `results/fid_predictions.csv` |
| `qa` | Table 5 — LLaMA-2-7B (vLLM), **session riêng** | **GPU T4×2** | 1–2h | `results/qa_predictions.csv` |
| `analysis` | toàn bộ bảng, hình, kiểm định | CPU/GPU | vài phút | `results/*.csv`, `*.png`, `summary.json` |

## 3. Hướng dẫn chạy trên Kaggle (2 lần chạy)

**Chuẩn bị (1 lần):**
1. Upload notebook: Kaggle → *Create → New Notebook → File → Import Notebook*.
2. Tạo Secret: *Add-ons → Secrets → Add* tên `HF_TOKEN` (token HuggingFace, quyền read). Tick bật cho notebook. Muốn chạy Table 5 thì tài khoản HF phải được Meta duyệt `meta-llama/Llama-2-7b-hf`.
3. Xác minh số điện thoại Kaggle (bắt buộc để bật Internet + GPU).

**Lần 1 — `PRESET = "run1"` (data + retrieval + fid + analysis):**
1. *Settings* → Accelerator **GPU T4 ×2**, Internet **ON**.
2. Cell đầu: `PRESET = "run1"`, `PREV_OUTPUT = None`.
3. *Save Version → Save & Run All (Commit)* → chờ 4–7h (có thể tắt tab).
4. Xem log: bảng title mapping, thống kê corpus, dòng **"Sanity — oracle EM"** của FiD (≳50% là đúng).

**Lần 2 — `PRESET = "run2"` (qa + analysis):**
1. Mở lại notebook (session mới). *Add Input → Your Work →* chọn notebook này, version của lần 1.
2. Copy đường dẫn thư mục `densex` trong sidebar Input, dán vào `PREV_OUTPUT`.
3. `PRESET = "run2"` → *Save & Run All (Commit)* → 1–2h.

**Kết quả:** tab *Output* của version cuối → thư mục `densex/results/` (tải về cả thư mục để làm báo cáo).

**Nếu bị ngắt giữa chừng:** dùng output của version bị lỗi làm Input, đặt `PREV_OUTPUT`, chạy lại cùng preset — các bước đã xong (corpus, từng retriever, từng cấu hình FiD) sẽ được bỏ qua.

In [ ]:
# ===== Chọn stage cần chạy =====
PRESET = "run1"          # "run1" = data+retrieval+fid+analysis | "run2" = qa+analysis | "custom" = tự đặt RUN bên dưới
PREV_OUTPUT = None       # None ở lần 1; lần 2 dán đường dẫn, vd "/kaggle/input/<ten-notebook>/densex"

PRESETS = {
    "run1": {"data": True,  "retrieval": True,  "fid": True,  "qa": False, "analysis": True},
    "run2": {"data": False, "retrieval": False, "fid": False, "qa": True,  "analysis": True},
}
RUN = PRESETS.get(PRESET) or {"data": True, "retrieval": False, "fid": False, "qa": False, "analysis": False}
print("PRESET:", PRESET, "→ RUN =", RUN)

import subprocess, sys
pkgs = ["huggingface_hub", "sentence-transformers", "orjson", "regex"]
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "-U", *pkgs], check=False)
if RUN["qa"]:
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "vllm"], check=False)   # chỉ ở session chạy qa

## 0. Config + seed

In [ ]:
import os, gc, json, gzip, time, math, random, bisect, html, unicodedata, shutil, string, re
from urllib.parse import unquote
from collections import defaultdict, Counter
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd, orjson, regex
from tqdm.auto import tqdm

SEED          = 42
N_SQUAD       = 500
N_NQ          = 500
N_TOTAL_PAGES = 8000
CAND_FACTOR   = 1.25          # lấy dư ứng viên để bù title bị miss (vẫn giữ thứ tự hoán vị ngẫu nhiên)
STRICT_NO_OVERLAP = True      # loại câu test trùng y hệt câu train (đảm bảo không leak)
N_DEMOS       = 4             # 4-shot như paper

K_LIST     = [1, 2, 5, 10, 20, 50, 100]
TOPN_UNITS = {"passage": 200, "sentence": 1000, "proposition": 2000}
MAX_LEN    = {"passage": 256, "sentence": 128, "proposition": 96}
L_WORDS    = [10, 25, 50, 100, 150, 200, 300, 400, 500]
EM_BUDGETS = [100, 500]
LLM_NAME   = "meta-llama/Llama-2-7b-hf"   # đúng paper; nếu chưa được duyệt có thể đổi model khác (ghi chú lại)
# FiD reader (Table 4) — checkpoint gốc của tác giả FiD, paper Appendix: "nq_reader_large"
FID_URL         = "https://dl.fbaipublicfiles.com/FiD/pretrained_models/nq_reader_large.tar.gz"
FID_K           = [5, 20]      # top-k passages như Table 4
FID_TEXT_MAXLEN = 200          # default của FiD (text_maxlength)
FID_ANS_MAXLEN  = 50           # default test_reader.py
FID_FP16        = False        # T5 fp16 dễ overflow → mặc định fp32 cho đúng

WORK  = "/kaggle/working/densex";  os.makedirs(WORK, exist_ok=True)
CACHE = "/tmp/factoidwiki";        os.makedirs(CACHE, exist_ok=True)
OUT   = f"{WORK}/results";         os.makedirs(OUT, exist_ok=True)
if PREV_OUTPUT and os.path.isdir(PREV_OUTPUT):
    shutil.copytree(PREV_OUTPUT, WORK, dirs_exist_ok=True,
                    ignore=None if RUN["retrieval"] else shutil.ignore_patterns("emb"))   # emb/ chỉ cần cho retrieval
    print("Loaded previous output from", PREV_OUTPUT)

REPOS = {"passage": "chentong00/factoid-wiki-passage",
         "sentence": "chentong00/factoid-wiki-sentence",
         "proposition": "chentong00/factoid-wiki"}
GRANS = ["passage", "sentence", "proposition"]
RETRIEVER_NAMES = ["SimCSE", "Contriever", "DPR", "GTR"]

def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); os.environ["PYTHONHASHSEED"] = str(s)
    try:
        import torch
        torch.manual_seed(s); torch.cuda.manual_seed_all(s)
        torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
    except ImportError: pass
set_seed()

if any(RUN[s] for s in ["retrieval", "fid", "qa"]):
    import torch
    assert torch.cuda.is_available(), "Stage retrieval/fid/qa cần GPU: Settings → Accelerator → GPU T4 x2"
    print("GPU:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN"); print("HF_TOKEN loaded")
except Exception:
    print("No HF_TOKEN secret")

def save_json(obj, path): json.dump(obj, open(path, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
def load_jsonl(p): return [json.loads(l) for l in open(p, encoding="utf-8")]
def write_jsonl(rows, p):
    with open(p, "w", encoding="utf-8") as f:
        for r in rows: f.write(json.dumps(r, ensure_ascii=False) + "\n")

## 1. Helpers
- ID FactoidWiki: `id[:15]` = page, `id[:25]` = passage → sentence/proposition map ngược về passage nguồn bằng id.
- `has_answer`: tokenizer kiểu `SimpleTokenizer` của DPR (cách paper chấm Recall@k).
- `normalize_answer`: chuẩn SQuAD (cho Exact Match).

In [ ]:
def page_id(uid):    return uid[:15]
def passage_id(uid): return uid[:25]

def parse_line(line):
    d = orjson.loads(line); c, m = d["contents"], d["metadata"]
    ts, te = m["title_span"]; ss, se = m["section_span"]; cs, ce = m["content_span"]
    return d["id"], c[ts:te], c, c[cs:ce], c[ss:se]      # id, title, contents, text, section

def norm_title(t):
    t = html.unescape(unquote(t)).replace("_", " ")
    return unicodedata.normalize("NFC", " ".join(t.split()))
def tkey(t):
    t = norm_title(t); return t[:1].upper() + t[1:]
def norm_q(q):
    return " ".join(q.lower().strip().rstrip("?").split())

_TOK = regex.compile(r"([\p{L}\p{N}\p{M}]+)|([^\p{Z}\p{C}])", flags=regex.IGNORECASE | regex.UNICODE | regex.MULTILINE)
def dpr_tokens(text):
    return [m.group().lower() for m in _TOK.finditer(unicodedata.normalize("NFD", text))]
def has_answer(ans_tok_list, text_tok):
    for a in ans_tok_list:
        n = len(a)
        if n == 0: continue
        for i in range(len(text_tok) - n + 1):
            if text_tok[i:i+n] == a: return True
    return False

def normalize_answer(s):
    s = s.lower(); s = "".join(ch for ch in s if ch not in set(string.punctuation))
    s = re.sub(r"\b(a|an|the)\b", " ", s); return " ".join(s.split())

def retry(fn, *a, tries=6, **kw):
    for i in range(tries):
        try: return fn(*a, **kw)
        except Exception:
            if i == tries - 1: raise
            time.sleep(2 ** i)

assert has_answer([dpr_tokens("5.5 degrees")], dpr_tokens("leaned at an angle of 5.5 degrees, but"))
assert not has_answer([dpr_tokens("Paris")], dpr_tokens("parisian cafe"))
assert normalize_answer("The Apollo 17!") == "apollo 17"
print("helpers OK")

# STAGE `data` (CPU)
## 2. Load test sets + train sets (chỉ dùng train để **check leak** và lấy **demo 4-shot**)

In [ ]:
if RUN["data"]:
    from datasets import load_dataset
    # ---------- SQuAD ----------
    sq_dev, sq_train = load_dataset("rajpurkar/squad", split="validation"), load_dataset("rajpurkar/squad", split="train")
    squad_test = [{"qid": f"squad-{ex['id']}", "dataset": "squad", "question": ex["question"],
                   "answers": list(dict.fromkeys(ex["answers"]["text"])), "gold_titles": [norm_title(ex["title"])]}
                  for ex in sq_dev]
    squad_train = [{"question": ex["question"], "answers": list(dict.fromkeys(ex["answers"]["text"])),
                    "title": norm_title(ex["title"])} for ex in sq_train]

    # ---------- NQ-open (test = split 'validation') ----------
    nqo_test  = load_dataset("google-research-datasets/nq_open", split="validation")
    nqo_train = load_dataset("google-research-datasets/nq_open", split="train")

    # ---------- official NQ dev → document.title ----------
    q2t_path = f"{WORK}/nq_question2title.json"
    if os.path.exists(q2t_path):
        q2title = json.load(open(q2t_path))
    else:
        try:    nq_full = load_dataset("google-research-datasets/natural_questions", "dev", split="validation", streaming=True)
        except Exception:
                nq_full = load_dataset("google-research-datasets/natural_questions", split="validation", streaming=True)
        q2title = {}
        for ex in tqdm(nq_full, desc="official NQ dev (title join)", total=7830):
            qtext = ex["question"]["text"] if isinstance(ex["question"], dict) else ex["question"]
            q2title.setdefault(norm_q(qtext), ex["document"]["title"])
        save_json(q2title, q2t_path)

    nq_test, nq_nojoin = [], 0
    for i, ex in enumerate(nqo_test):
        t = q2title.get(norm_q(ex["question"]))
        if t is None: nq_nojoin += 1; continue
        nq_test.append({"qid": f"nq-test-{i}", "dataset": "nq", "question": ex["question"],
                        "answers": list(ex["answer"]), "gold_titles": [norm_title(t)]})
    nq_train = [{"question": ex["question"], "answers": list(ex["answer"])} for ex in nqo_train]

    print(f"SQuAD dev : {len(squad_test):6d} q | {len({q['gold_titles'][0] for q in squad_test})} titles")
    print(f"NQ-open test: {len(nqo_test)} q | join được title: {len(nq_test)} | không join được: {nq_nojoin} "
          f"| {len({q['gold_titles'][0] for q in nq_test})} titles")

## 3. Leakage check
1. Câu test trùng **y hệt** câu train (SQuAD train / NQ-open train) → loại nếu `STRICT_NO_OVERLAP`.
2. Title SQuAD dev ∩ train (theo thiết kế SQuAD phải = 0).
3. Retriever: không train/fine-tune gì; DPR-multiset đã thấy **train split** NQ & SQuAD (paper cũng gạch chân các ô này) → không trùng câu, chỉ "quen phân phối".
4. Demo 4-shot lấy từ **train**, không trùng câu test.

In [ ]:
if RUN["data"]:
    sq_train_q = {norm_q(x["question"]) for x in squad_train}
    nq_train_q = {norm_q(x["question"]) for x in nq_train}
    sq_overlap = [q for q in squad_test if norm_q(q["question"]) in sq_train_q]
    nq_overlap = [q for q in nq_test if norm_q(q["question"]) in nq_train_q]
    sq_title_overlap = {q["gold_titles"][0] for q in squad_test} & {x["title"] for x in squad_train}
    LEAK = {
        "squad_test_questions": len(squad_test), "squad_exact_overlap_with_train": len(sq_overlap),
        "squad_dev_train_title_overlap": len(sq_title_overlap),
        "nq_test_questions": len(nq_test), "nq_exact_overlap_with_train": len(nq_overlap),
        "strict_removed": STRICT_NO_OVERLAP,
        "retriever_training_note": {
            "SimCSE": "unsupervised (Wikipedia sentences) — không thấy QA labels",
            "Contriever": "unsupervised contrastive (Wikipedia + CC) — không thấy QA labels",
            "DPR-multiset": "supervised trên TRAIN của NQ, TQA, WebQ, SQuAD → seen-distribution (paper gạch chân)",
            "GTR": "pre-train forum QA + fine-tune MS MARCO",
        },
    }
    if STRICT_NO_OVERLAP:
        bad = {q["qid"] for q in sq_overlap + nq_overlap}
        squad_test = [q for q in squad_test if q["qid"] not in bad]
        nq_test    = [q for q in nq_test if q["qid"] not in bad]
    print(json.dumps(LEAK, indent=1, ensure_ascii=False))

## 4. Index shard FactoidWiki (chỉ đọc dòng đầu mỗi shard)

In [ ]:
if RUN["data"]:
    from huggingface_hub import HfFileSystem, hf_hub_download
    fs = HfFileSystem()

    def list_shards(repo):
        files = retry(fs.ls, f"datasets/{repo}/data", detail=True)
        files = sorted((f for f in files if f["name"].endswith(".jsonl")), key=lambda f: f["name"])
        return [(f["name"].split("/data/")[-1], f["size"]) for f in files]
    def first_record(repo, fname):
        def _read():
            with fs.open(f"datasets/{repo}/data/{fname}", "rb", block_size=256 * 1024) as f:
                return parse_line(f.readline())[:2]
        return retry(_read)

    idx_path = f"{WORK}/shard_index.json"
    if os.path.exists(idx_path): SHARDS = json.load(open(idx_path))
    else:
        SHARDS = {}
        for g in GRANS:
            lst = list_shards(REPOS[g]); SHARDS[g] = {"files": [n for n, _ in lst], "sizes": [s for _, s in lst]}
        with ThreadPoolExecutor(8) as ex:
            recs = list(tqdm(ex.map(lambda fn: first_record(REPOS["passage"], fn), SHARDS["passage"]["files"]),
                             total=len(SHARDS["passage"]["files"]), desc="index passage shards"))
        SHARDS["passage"]["first_id"] = [r[0] for r in recs]; SHARDS["passage"]["first_title"] = [r[1] for r in recs]
        save_json(SHARDS, idx_path)

    P_FILES, P_FIRST, P_TITLES = (SHARDS["passage"][k] for k in ["files", "first_id", "first_title"])
    for g in GRANS: print(f"{g:12s}: {len(SHARDS[g]['files'])} shards, {sum(SHARDS[g]['sizes'])/1e9:.1f} GB")
    print("ids monotonic:", P_FIRST == sorted(P_FIRST),
          "| title order violations:", sum(P_TITLES[i] > P_TITLES[i+1] for i in range(len(P_TITLES)-1)))

    # alignment 3 granularity
    ALIGNED = True
    for g in ["sentence", "proposition"]:
        if SHARDS[g]["files"] != P_FILES: ALIGNED = False; continue
        for i in sorted(set([0, len(P_FILES)//2, len(P_FILES)-1] + random.Random(SEED).sample(range(len(P_FILES)), min(12, len(P_FILES))))):
            if page_id(first_record(REPOS[g], P_FILES[i])[0]) != page_id(P_FIRST[i]): ALIGNED = False
    if not ALIGNED and "first_id" not in SHARDS["sentence"]:
        for g in ["sentence", "proposition"]:
            with ThreadPoolExecutor(8) as ex:
                SHARDS[g]["first_id"] = [r[0] for r in tqdm(ex.map(lambda fn: first_record(REPOS[g], fn), SHARDS[g]["files"]),
                                                            total=len(SHARDS[g]["files"]), desc=f"index {g}")]
        save_json(SHARDS, idx_path)
    print("aligned:", ALIGNED)

    def unit_shards_for(g, i):
        if ALIGNED: return [SHARDS[g]["files"][i]]
        lo = page_id(P_FIRST[i]); hi = page_id(P_FIRST[i+1]) if i + 1 < len(P_FIRST) else "~"
        firsts = [page_id(x) for x in SHARDS[g]["first_id"]]
        a = max(0, bisect.bisect_right(firsts, lo) - 1); b = bisect.bisect_left(firsts, hi)
        return SHARDS[g]["files"][a:max(b, a + 1)]
    def shard_of_title(t): return max(0, bisect.bisect_right(P_TITLES, t) - 1)
    def get_shard(g, fname):
        return retry(hf_hub_download, REPOS[g], f"data/{fname}", repo_type="dataset", local_dir=f"{CACHE}/{g}")

## 5. Chọn câu hỏi (random, seed cố định) + verify title trong FactoidWiki

Hoán vị ngẫu nhiên tập test → duyệt theo thứ tự đó, câu nào **map được gold title** thì nhận, đủ 500 thì dừng. Không chọn theo shard → không lệch phân phối. Title được verify bằng cách đọc chính `title` của từng page trong shard (exact, rồi casefold).

In [ ]:
if RUN["data"]:
    page_title, page_shard, shard_pages = {}, {}, defaultdict(list)
    key2page, fold2page, scanned = {}, {}, set()

    def scan(shards):
        for i in tqdm(sorted(set(shards) - scanned), desc="scan passage shards", leave=False):
            path = get_shard("passage", P_FILES[i])
            with open(path, "rb") as f:
                for line in f:
                    uid, title, *_ = parse_line(line); pg = page_id(uid)
                    if pg in page_title: continue
                    page_title[pg] = title; page_shard[pg] = i; shard_pages[i].append(pg)
                    k = tkey(title); key2page.setdefault(k, pg); fold2page.setdefault(k.casefold(), pg)
            os.remove(path); scanned.add(i)

    def resolve(t):
        k = tkey(t)
        if k in key2page: return key2page[k], "exact"
        if k.casefold() in fold2page: return fold2page[k.casefold()], "casefold"
        return None, "missing"

    def select(pool, n, name):
        order = list(pool); random.Random(SEED).shuffle(order)
        chosen, checked, ptr = [], [], 0
        while len(chosen) < n and ptr < len(order):
            batch = order[ptr: ptr + max(1, math.ceil((n - len(chosen)) * CAND_FACTOR))]; ptr += len(batch)
            scan({shard_of_title(tkey(t)) for q in batch for t in q["gold_titles"]})
            for q in batch:
                res = [resolve(t) for t in q["gold_titles"]]
                q["title_status"] = [r[1] for r in res]
                q["gold_pages"] = sorted({r[0] for r in res if r[0]})
                checked.append(q)
                if q["gold_pages"] and len(chosen) < n: chosen.append(q)
            print(f"{name}: checked {len(checked)} | chosen {len(chosen)} | shards scanned {len(scanned)}")
        return chosen, checked

    sq_chosen, sq_checked = select(squad_test, N_SQUAD, "SQuAD")
    nq_chosen, nq_checked = select(nq_test, N_NQ, "NQ")
    QUESTIONS = sq_chosen + nq_chosen

    rows = []
    for name, checked, chosen in [("SQuAD", sq_checked, sq_chosen), ("NQ", nq_checked, nq_chosen)]:
        st = {}
        for q in checked:
            for t, s in zip(q["gold_titles"], q["title_status"]): st[t] = s
        c = Counter(st.values())
        rows.append({"dataset": name, "questions checked": len(checked), "questions chosen": len(chosen),
                     "unique titles checked": len(st), "exact": c["exact"], "casefold": c["casefold"], "missing": c["missing"],
                     "title match %": round(100 * (c["exact"] + c["casefold"]) / max(1, len(st)), 1),
                     "gold pages (chosen)": len({p for q in chosen for p in q["gold_pages"]})})
        pd.Series([t for t, s in st.items() if s == "missing"], name="missing_title").to_csv(f"{OUT}/missing_titles_{name}.csv", index=False)
    TITLE_REPORT = pd.DataFrame(rows); TITLE_REPORT.to_csv(f"{OUT}/title_mapping_report.csv", index=False); display(TITLE_REPORT)

## 6. Gold pages + distractors (đã loại gold) + demo 4-shot
Distractor lấy ngẫu nhiên (seed) từ các page nằm trong **shard của gold pages**, **loại toàn bộ gold page** → tổng đúng `N_TOTAL_PAGES`.

In [ ]:
if RUN["data"]:
    gold_set = {p for q in QUESTIONS for p in q["gold_pages"]}
    gold_shards = sorted({page_shard[p] for p in gold_set})
    pool = sorted(p for i in gold_shards for p in shard_pages[i] if p not in gold_set)
    n_distr = N_TOTAL_PAGES - len(gold_set)
    distractors = sorted(random.Random(SEED).sample(pool, n_distr))
    assert not (set(distractors) & gold_set)
    CORPUS_PAGES = gold_set | set(distractors)
    assert len(CORPUS_PAGES) == N_TOTAL_PAGES

    # demos 4-shot từ TRAIN (không trùng test), answer ngắn
    test_q = {norm_q(q["question"]) for q in QUESTIONS}
    def pick_demos(train, name):
        cand = [x for x in train if norm_q(x["question"]) not in test_q and x["answers"] and len(x["answers"][0].split()) <= 4]
        return [{"qid": f"demo-{name}-{i}", "dataset": name, "question": x["question"], "answers": x["answers"]}
                for i, x in enumerate(random.Random(SEED).sample(cand, N_DEMOS))]
    DEMOS = pick_demos(squad_train, "squad") + pick_demos(nq_train, "nq")

    sq_pages = {p for q in QUESTIONS if q["dataset"] == "squad" for p in q["gold_pages"]}
    nq_pages = {p for q in QUESTIONS if q["dataset"] == "nq" for p in q["gold_pages"]}
    PAGE_SUMMARY = {"gold_pages_total": len(gold_set), "gold_squad": len(sq_pages), "gold_nq": len(nq_pages),
                    "gold_shared_squad_nq": len(sq_pages & nq_pages), "distractor_pages": len(distractors),
                    "distractor_pool_size": len(pool), "corpus_pages": len(CORPUS_PAGES), "shards_used": len(gold_shards)}
    print(json.dumps(PAGE_SUMMARY, indent=1))
    est = sum(SHARDS[g]["sizes"][i] for g in GRANS for i in gold_shards) / 1e9
    print(f"Pass 2 sẽ tải ~{est:.1f} GB (từng shard, xử lý xong xoá ngay)")

## 7. Pass 2 — trích passage / sentence / proposition của 8,000 pages (resume được)

In [ ]:
if RUN["data"]:
    PARTS = f"{WORK}/corpus/parts"
    for g in GRANS: os.makedirs(f"{PARTS}/{g}", exist_ok=True)
    def dump(g, fname, path):
        tmp = f"{PARTS}/{g}/{fname}.tmp"
        with open(path, "rb") as f, open(tmp, "w", encoding="utf-8") as o:
            for line in f:
                uid, title, contents, text, section = parse_line(line)
                if page_id(uid) in CORPUS_PAGES:
                    o.write(json.dumps({"id": uid, "pid": passage_id(uid), "page": page_id(uid), "title": title,
                                        "section": section, "contents": contents, "text": text}, ensure_ascii=False) + "\n")
        os.replace(tmp, f"{PARTS}/{g}/{fname}")
    for i in tqdm(gold_shards, desc="pass2"):
        jobs = [("passage", P_FILES[i])] + [(g, fn) for g in ["sentence", "proposition"] for fn in unit_shards_for(g, i)]
        for g, fn in jobs:
            if os.path.exists(f"{PARTS}/{g}/{fn}"): continue
            path = get_shard(g, fn); dump(g, fn, path); os.remove(path)

    for g in GRANS:   # gộp + dedupe + sort
        seen, rows = set(), []
        for fn in sorted(os.listdir(f"{PARTS}/{g}")):
            if fn.endswith(".tmp"): continue
            for r in load_jsonl(f"{PARTS}/{g}/{fn}"):
                if r["id"] not in seen: seen.add(r["id"]); rows.append(r)
        rows.sort(key=lambda r: r["id"]); write_jsonl(rows, f"{WORK}/corpus/{g}.jsonl")
        print(g, len(rows))
    shutil.rmtree(PARTS)

    for q in QUESTIONS: q.pop("cand_shards", None)
    write_jsonl(QUESTIONS, f"{WORK}/questions.jsonl"); write_jsonl(DEMOS, f"{WORK}/demos.jsonl")
    save_json({"seed": SEED, "config": {"N_SQUAD": N_SQUAD, "N_NQ": N_NQ, "N_TOTAL_PAGES": N_TOTAL_PAGES,
               "CAND_FACTOR": CAND_FACTOR, "STRICT_NO_OVERLAP": STRICT_NO_OVERLAP},
               "leak": LEAK, "pages": PAGE_SUMMARY, "gold_pages": sorted(gold_set), "distractors": distractors,
               "aligned_shards": ALIGNED}, f"{WORK}/corpus_meta.json")
    print("Stage data xong → Save Version")

# Load corpus (dùng chung cho mọi stage sau)
## 8. Thống kê corpus mới

In [ ]:
assert os.path.exists(f"{WORK}/corpus/passage.jsonl"), \
    "Chưa có corpus: bật stage data, hoặc đặt PREV_OUTPUT trỏ tới output của lần chạy trước"
UNITS = {g: load_jsonl(f"{WORK}/corpus/{g}.jsonl") for g in GRANS}
QUESTIONS = load_jsonl(f"{WORK}/questions.jsonl"); DEMOS = load_jsonl(f"{WORK}/demos.jsonl")
META = json.load(open(f"{WORK}/corpus_meta.json"))
GOLD = set(META["gold_pages"])
PIDS = [u["id"] for u in UNITS["passage"]]; PID2IDX = {p: i for i, p in enumerate(PIDS)}
UNIT_PIDX = {g: np.array([PID2IDX[u["pid"]] for u in UNITS[g]], dtype=np.int32) for g in GRANS}   # lỗi KeyError = unit mồ côi
PAGE_OF_PID = np.array([u["page"] for u in UNITS["passage"]])

print("=== Pages ==="); print(json.dumps(META["pages"], indent=1))
print("=== Leakage ==="); print(json.dumps({k: v for k, v in META["leak"].items() if k != "retriever_training_note"}, indent=1))

rows = []
for g in GRANS:
    U = UNITS[g]; w = np.array([len(u["text"].split()) for u in U])
    per_page = Counter(u["page"] for u in U); per_pass = Counter(u["pid"] for u in U)
    is_gold = np.array([u["page"] in GOLD for u in U])
    rows.append({"granularity": g, "#units": len(U), "#pages": len(per_page),
                 "units/page (mean)": round(np.mean(list(per_page.values())), 1),
                 "units/passage (mean)": round(np.mean(list(per_pass.values())), 2),
                 "avg words": round(w.mean(), 1), "median words": int(np.median(w)),
                 "paper avg words": {"passage": 58.5, "sentence": 21.0, "proposition": 11.2}[g],
                 "% units from gold pages": round(100 * is_gold.mean(), 1)})
CORPUS_STATS = pd.DataFrame(rows); CORPUS_STATS.to_csv(f"{OUT}/corpus_stats.csv", index=False); display(CORPUS_STATS)

pp = Counter(u["page"] for u in UNITS["passage"])
print("passages/page — gold: %.1f | distractor: %.1f" % (np.mean([pp[p] for p in GOLD]), np.mean([pp[p] for p in pp if p not in GOLD])))
print("questions:", Counter(q["dataset"] for q in QUESTIONS))

## 9. Answer index (dùng cho mọi metric)
Inverted index trên token passage → với mỗi câu hỏi tính sẵn **tập passage chứa answer** (`ANS_P`). Từ đó: Recall@k, answer coverage trong gold page, độ "hiếm" của answer (chuyển thể Figure 3).

In [ ]:
P_TOK = [dpr_tokens(u["text"]) for u in tqdm(UNITS["passage"], desc="tokenize passages")]
INV = defaultdict(set)
for i, toks in enumerate(P_TOK):
    for t in set(toks): INV[t].add(i)

def passages_with_answer(answers):
    hit = set()
    for a in answers:
        at = dpr_tokens(a)
        if not at: continue
        cand = set.intersection(*[INV.get(t, set()) for t in sorted(set(at), key=lambda t: len(INV.get(t, ())))])
        hit |= {i for i in cand if has_answer([at], P_TOK[i])}
    return hit

for q in tqdm(QUESTIONS + DEMOS, desc="answer index"):
    q["ans_tok"] = [dpr_tokens(a) for a in q["answers"]]
    q["ans_p"] = passages_with_answer(q["answers"])
    q["ans_freq"] = len(q["ans_p"])
    q["answer_in_gold"] = any(PAGE_OF_PID[i] in set(q.get("gold_pages", [])) for i in q["ans_p"])

cov = pd.DataFrame([{"dataset": q["dataset"], "answer_in_gold_page": q["answer_in_gold"], "answer_in_corpus": q["ans_freq"] > 0,
                     "ans_freq": q["ans_freq"]} for q in QUESTIONS])
COVERAGE = cov.groupby("dataset").agg(answer_in_gold_page_pct=("answer_in_gold_page", lambda s: round(100*s.mean(), 1)),
                                      answer_in_corpus_pct=("answer_in_corpus", lambda s: round(100*s.mean(), 1)),
                                      median_passages_with_answer=("ans_freq", "median"))
COVERAGE.to_csv(f"{OUT}/answer_coverage.csv"); display(COVERAGE)

# STAGE `retrieval` (GPU)
## 10. Retrievers — vì sao 4 cái này
Đúng 4 retriever của paper (Appendix: checkpoint HF) và chia 2 nhóm để trả lời câu hỏi chính: proposition có lợi cho retriever **chưa từng train trên QA** (SimCSE, Contriever) nhiều hơn retriever **đã supervised trên cặp question–passage** (DPR, GTR) không?

| Retriever | Checkpoint | Pooling | Sim |
|---|---|---|---|
| SimCSE | `princeton-nlp/unsup-simcse-bert-base-uncased` | CLS | cosine |
| Contriever | `facebook/contriever` | mean | dot |
| DPR | `facebook/dpr-{question,ctx}_encoder-multiset-base` | pooler | dot |
| GTR | `sentence-transformers/gtr-t5-base` | ST | cosine |

Tối ưu Kaggle: 1 replica/GPU (T4×2 chạy song song), sort theo độ dài, fp16 (GTR/T5 chạy fp32 để tránh overflow), `max_length` theo granularity, cache embedding `.npy`, search bằng matmul trên GPU.

In [ ]:
if RUN["retrieval"]:
    import torch, torch.nn.functional as F
    from transformers import AutoTokenizer, AutoModel, DPRQuestionEncoder, DPRContextEncoder
    from sentence_transformers import SentenceTransformer
    NG = torch.cuda.device_count(); DEVICES = [f"cuda:{i}" for i in range(NG)] or ["cpu"]
    print("GPUs:", [torch.cuda.get_device_name(i) for i in range(NG)])
    EMB = f"{WORK}/emb"; os.makedirs(EMB, exist_ok=True)

    def _parallel(n_items, work):
        with ThreadPoolExecutor(len(DEVICES)) as ex: return list(ex.map(work, range(len(DEVICES))))

    class HFEncoder:
        def __init__(self, qn, cn, pooling, normalize, dpr=False):
            self.pooling, self.normalize = pooling, normalize
            self.qt, self.ct = AutoTokenizer.from_pretrained(qn), AutoTokenizer.from_pretrained(cn)
            self.reps = []
            for d in DEVICES:
                if dpr: qm, cm = DPRQuestionEncoder.from_pretrained(qn), DPRContextEncoder.from_pretrained(cn)
                else:   qm = cm = AutoModel.from_pretrained(qn)
                qm = qm.to(d).half().eval(); cm = qm if cm is qm else cm.to(d).half().eval()
                self.reps.append((d, qm, cm))
        @torch.no_grad()
        def _enc(self, texts, side, max_len, bs):
            order = np.argsort([-len(t) for t in texts]); batches = [order[s:s+bs] for s in range(0, len(order), bs)]
            tok = self.qt if side == "q" else self.ct
            @torch.inference_mode()        # grad mode là thread-local → phải đặt trong từng thread
            def work(r):
                d, qm, cm = self.reps[r]; m = qm if side == "q" else cm; res = []
                for b in tqdm(batches[r::len(self.reps)], leave=False, position=r):
                    x = tok([texts[i] for i in b], padding=True, truncation=True, max_length=max_len, return_tensors="pt").to(d)
                    o = m(**x)
                    if self.pooling == "pooler": e = o.pooler_output
                    elif self.pooling == "cls":  e = o.last_hidden_state[:, 0]
                    else:
                        mk = x["attention_mask"].unsqueeze(-1).to(o.last_hidden_state.dtype)
                        e = (o.last_hidden_state * mk).sum(1) / mk.sum(1)
                    e = e.float()
                    if self.normalize: e = F.normalize(e, dim=-1)
                    res.append((b, e.cpu().numpy()))
                return res
            parts = _parallel(len(texts), work)
            dim = parts[0][0][1].shape[1]; out = np.zeros((len(texts), dim), dtype=np.float32)
            for p in parts:
                for b, e in p: out[b] = e
            return out
        def encode_queries(self, texts): return self._enc(texts, "q", 64, 256)
        def encode_corpus(self, texts, g): return self._enc(texts, "c", MAX_LEN[g], 256 if g != "passage" else 128)

    class GTREncoder:
        def __init__(self, name="sentence-transformers/gtr-t5-base"):
            self.reps = [SentenceTransformer(name, device=d) for d in DEVICES]       # fp32 (T5 fp16 dễ overflow)
        def _enc(self, texts, max_len, bs):
            for m in self.reps: m.max_seq_length = max_len
            order = np.argsort([-len(t) for t in texts])
            def work(r):
                idx = order[r::len(self.reps)]
                return idx, self.reps[r].encode([texts[i] for i in idx], batch_size=bs, normalize_embeddings=True,
                                               convert_to_numpy=True, show_progress_bar=(r == 0))
            parts = _parallel(len(texts), work)
            out = np.zeros((len(texts), parts[0][1].shape[1]), dtype=np.float32)
            for idx, e in parts: out[idx] = e
            return out
        def encode_queries(self, texts): return self._enc(texts, 64, 256)
        def encode_corpus(self, texts, g): return self._enc(texts, MAX_LEN[g], 256 if g != "passage" else 128)

    RETRIEVERS = {
        "SimCSE":     lambda: HFEncoder("princeton-nlp/unsup-simcse-bert-base-uncased", "princeton-nlp/unsup-simcse-bert-base-uncased", "cls", True),
        "Contriever": lambda: HFEncoder("facebook/contriever", "facebook/contriever", "mean", False),
        "DPR":        lambda: HFEncoder("facebook/dpr-question_encoder-multiset-base", "facebook/dpr-ctx_encoder-multiset-base", "pooler", False, dpr=True),
        "GTR":        lambda: GTREncoder(),
    }

## 11. Retrieval (test queries + demo queries)
Unit → passage nguồn: score passage = **max** score unit (paper, Lee et al. 2021b). Lưu ranked units (cho Recall@l words và EM@l) và thời gian / dung lượng index.

In [ ]:
if RUN["retrieval"]:
    @torch.no_grad()
    def search(q, c, topn, qbs=128):
        C = torch.from_numpy(c).to(DEVICES[0]).half(); S, I = [], []
        torch.cuda.synchronize(); t0 = time.perf_counter()
        for s in range(0, len(q), qbs):
            v, i = (torch.from_numpy(q[s:s+qbs]).to(DEVICES[0]).half() @ C.T).float().topk(min(topn, C.shape[0]), dim=1)
            S.append(v.cpu().numpy()); I.append(i.cpu().numpy().astype(np.int32))
        torch.cuda.synchronize(); dt = time.perf_counter() - t0
        del C; torch.cuda.empty_cache()
        return np.concatenate(S), np.concatenate(I), dt

    ALL_Q = QUESTIONS + DEMOS; q_texts = [q["question"] for q in ALL_Q]
    RANK_DIR = f"{WORK}/ranked"; os.makedirs(RANK_DIR, exist_ok=True); os.makedirs(EMB, exist_ok=True)
    eff_path = f"{OUT}/efficiency.csv"
    EFF = pd.read_csv(eff_path).to_dict("records") if os.path.exists(eff_path) else []
    for rname in RETRIEVER_NAMES:
        if all(os.path.exists(f"{RANK_DIR}/{rname}_{g}.npz") for g in GRANS): print("skip", rname); continue
        set_seed(); enc = RETRIEVERS[rname]()
        q_emb = enc.encode_queries(q_texts).astype(np.float16)
        for g in GRANS:
            if os.path.exists(f"{RANK_DIR}/{rname}_{g}.npz"): continue
            ep = f"{EMB}/{rname}_{g}.npy"; t0 = time.perf_counter()
            if os.path.exists(ep): c_emb = np.load(ep); t_enc = float("nan")
            else:
                c_emb = enc.encode_corpus([u["contents"] for u in UNITS[g]], g)
                assert np.isfinite(c_emb).all(), f"NaN/inf in {rname}/{g}"
                c_emb = c_emb.astype(np.float16); np.save(ep, c_emb); t_enc = time.perf_counter() - t0
            S, I, t_search = search(q_emb, c_emb, TOPN_UNITS[g])
            np.savez_compressed(f"{RANK_DIR}/{rname}_{g}.npz", I=I, S=S.astype(np.float16), qids=np.array([q["qid"] for q in ALL_Q]))
            EFF.append({"retriever": rname, "granularity": g, "#units": len(UNITS[g]), "dim": c_emb.shape[1],
                        "index_MB_fp16": round(c_emb.nbytes / 2**20, 1), "encode_sec": round(t_enc, 1),
                        "search_ms_per_query": round(1000 * t_search / len(q_texts), 3)})
            pd.DataFrame(EFF).to_csv(eff_path, index=False)
            print(f"{rname}/{g}: {len(UNITS[g])} units | encode {t_enc:.0f}s | search {t_search:.2f}s")
            del c_emb; gc.collect()
        del enc; gc.collect(); torch.cuda.empty_cache()
    display(pd.DataFrame(EFF))

# STAGE `fid` (GPU)
## 11b. Table 4 — FiD reader (`nq_reader_large`, T5-large train trên NQ)
Paper Appendix: *"T5-large size Fusion-in-decoder model (nq_reader_large) released by the authors in facebookresearch/FiD"*.
- Code gốc FiD cần transformers 3.0.2 / torch 1.6 → **chỉ lấy trọng số**, đổi tên key (`encoder.encoder.` → `encoder.`, bỏ `.module.`) rồi nạp vào `T5ForConditionalGeneration` mới. Phần *fusion* viết lại: encode từng `question + passage` riêng → nối encoder output → decode.
- Input giống FiD: `question: … title: … context: …`, mỗi passage cắt 200 token, greedy, `max_length=50`.
- Đầu vào là **top-5 / top-20 passage** (unit đã map về passage nguồn — như Recall@k), đúng caption Table 4.
- **Sanity check** trước khi chạy: đưa passage chứa answer (oracle) → EM phải cao; nếu ≈0 là nạp trọng số sai → dừng.
- Resume theo từng cấu hình (`results/fid_parts/`), chạy song song 2 GPU.

In [ ]:
if RUN["fid"]:
    import torch, glob, subprocess
    from transformers import T5ForConditionalGeneration, T5Config, AutoTokenizer
    from transformers.modeling_outputs import BaseModelOutput
    FID_DIR = "/tmp/fid"                       # ngoài /kaggle/working → không chiếm 20GB output
    def _ck(): return glob.glob(f"{FID_DIR}/**/pytorch_model.bin", recursive=True)
    if not _ck():
        os.makedirs(FID_DIR, exist_ok=True)
        subprocess.run(f"wget -qO- {FID_URL} | tar xz -C {FID_DIR}", shell=True, check=True)
    raw = torch.load(_ck()[0], map_location="cpu", weights_only=False)
    FID_SD = {re.sub(r"^encoder\.encoder\.", "encoder.", re.sub(r"^module\.", "", k)).replace(".module.", "."): v
              for k, v in raw.items()}
    del raw
    assert "shared.weight" in FID_SD, "checkpoint FiD không có shared.weight — kiểm tra lại file"
    FTOK = AutoTokenizer.from_pretrained("google-t5/t5-large")
    FCFG = T5Config.from_pretrained("google-t5/t5-large")
    TIED = {"encoder.embed_tokens.weight", "decoder.embed_tokens.weight", "lm_head.weight"}

    def load_fid(dev):
        m = T5ForConditionalGeneration(FCFG)
        missing, unexpected = m.load_state_dict(FID_SD, strict=False)
        bad = [k for k in missing if k not in TIED]
        assert not bad and not unexpected, f"FiD key mismatch: missing={bad[:5]} unexpected={unexpected[:5]}"
        m.tie_weights(); m = m.to(dev).eval()
        return m.half() if FID_FP16 else m

    PASSAGE = UNITS["passage"]
    def fid_text(question, p):
        return f"question: {question} title: {PASSAGE[p]['title']} context: {PASSAGE[p]['text']}"

    @torch.no_grad()
    def fid_generate(m, dev, items, bs):
        # items: list[(question, [passage_idx ...])], cùng số passage n
        preds = []
        for s in range(0, len(items), bs):
            chunk = items[s:s+bs]; n = len(chunk[0][1])
            x = FTOK([fid_text(q, p) for q, ps in chunk for p in ps], padding="max_length", truncation=True,
                     max_length=FID_TEXT_MAXLEN, return_tensors="pt").to(dev)
            h = m.encoder(input_ids=x.input_ids, attention_mask=x.attention_mask).last_hidden_state
            h = h.reshape(len(chunk), n * FID_TEXT_MAXLEN, h.size(-1))              # fusion
            am = x.attention_mask.reshape(len(chunk), n * FID_TEXT_MAXLEN)
            out = m.generate(encoder_outputs=BaseModelOutput(last_hidden_state=h), attention_mask=am,
                             max_length=FID_ANS_MAXLEN, num_beams=1, do_sample=False)
            preds += FTOK.batch_decode(out, skip_special_tokens=True)
        return preds

    def em(pred, answers): return float(any(normalize_answer(pred) == normalize_answer(a) for a in answers))
    def pad_to(ps, n): return (list(ps) + [ps[-1]] * n)[:n]

    QID2I = {q["qid"]: i for i, q in enumerate(QUESTIONS + DEMOS)}
    def top_passages(I_row, g, k):
        seen, out = set(), []
        for j in I_row:
            p = int(UNIT_PIDX[g][j])
            if p not in seen:
                seen.add(p); out.append(p)
                if len(out) == k: break
        return pad_to(out, k)

    NG = max(1, torch.cuda.device_count()); DEVS = [f"cuda:{i}" for i in range(NG)] if torch.cuda.is_available() else ["cpu"]
    set_seed(); FID_MODELS = [load_fid(d) for d in DEVS]
    print("FiD loaded on", DEVS, "| dtype", next(FID_MODELS[0].parameters()).dtype)

    # ---- sanity check: oracle passages (chứa answer) phải cho EM cao ----
    oracle = [q for q in QUESTIONS if q["dataset"] == "nq" and q["ans_p"]][:100]
    items = [(q["question"], pad_to(sorted(q["ans_p"])[:5], 5)) for q in oracle]
    ora = fid_generate(FID_MODELS[0], DEVS[0], items, 16)
    ora_em = 100 * np.mean([em(p, q["answers"]) for p, q in zip(ora, oracle)])
    print(f"Sanity — oracle EM trên {len(oracle)} câu NQ: {ora_em:.1f}%  | ví dụ:",
          [(q["question"], p, q["answers"][:2]) for q, p in list(zip(oracle, ora))[:3]])
    save_json({"oracle_nq_n": len(oracle), "oracle_nq_em": round(ora_em, 1)}, f"{OUT}/fid_sanity.json")
    assert ora_em > 20, "EM oracle quá thấp → trọng số FiD nạp sai, dừng lại kiểm tra"

    # ---- chạy 4 retriever × 3 granularity × {top-5, top-20} ----
    PARTS = f"{OUT}/fid_parts"; os.makedirs(PARTS, exist_ok=True)
    jobs = [(r, g, k) for r in RETRIEVER_NAMES for g in GRANS for k in FID_K
            if not os.path.exists(f"{PARTS}/{r}_{g}_k{k}.csv")]
    print(len(jobs), "cấu hình cần chạy")

    def worker(w):
        m, dev = FID_MODELS[w], DEVS[w]; log = []
        for r, g, k in jobs[w::len(DEVS)]:
            t0 = time.perf_counter()
            I = np.load(f"{WORK}/ranked/{r}_{g}.npz")["I"]
            items = [(q["question"], top_passages(I[QID2I[q["qid"]]], g, k)) for q in QUESTIONS]
            preds = fid_generate(m, dev, items, bs=16 if k <= 5 else 4)
            pd.DataFrame([{"qid": q["qid"], "dataset": q["dataset"], "retriever": r, "granularity": g, "k": k,
                           "pred": p, "EM": em(p, q["answers"])} for q, p in zip(QUESTIONS, preds)]
                         ).to_csv(f"{PARTS}/{r}_{g}_k{k}.csv", index=False)
            log.append(f"[{dev}] {r}/{g}/top{k}: {time.perf_counter()-t0:.0f}s")
            print(log[-1], flush=True)
        return log
    with ThreadPoolExecutor(len(DEVS)) as ex: list(ex.map(worker, range(len(DEVS))))

    FIDP = pd.concat([pd.read_csv(f"{PARTS}/{f}") for f in sorted(os.listdir(PARTS))], ignore_index=True)
    FIDP.to_csv(f"{OUT}/fid_predictions.csv", index=False)
    del FID_MODELS, FID_SD; gc.collect(); torch.cuda.empty_cache()     # giải phóng GPU trước stage qa (nếu chạy chung)
    print("FiD xong:", FIDP.shape)

# STAGE `qa` (GPU, session mới có vLLM)
## 12. Open-domain QA — LLaMA-2-7B, 4-shot, EM@l tokens (Table 5)
Prompt theo Figure 9 của paper. Context = các unit retrieve được (không map về passage), cắt đúng **l token** (tokenizer LLaMA, tính cả dòng `Title: … Passage: …`). Demo 4-shot lấy từ train, context của demo cũng retrieve bằng cùng retriever/granularity/budget. Thêm baseline **closed-book**. Greedy decoding → deterministic. vLLM + prefix caching + tensor-parallel 2 GPU.

In [ ]:
if RUN["qa"]:
    import torch
    from vllm import LLM, SamplingParams
    from transformers import AutoTokenizer
    LTOK = AutoTokenizer.from_pretrained(LLM_NAME)
    INSTR1 = "Refer to the passages below and answer the following question with just a few words."
    INSTR2 = "Refer to the context above and answer the following question with just a few words."
    QID2I = {q["qid"]: i for i, q in enumerate(QUESTIONS + DEMOS)}
    _len_cache = {}

    def ctx_lines(g, unit_ids, budget):
        lines, used = [], 0
        for j in unit_ids:
            u = UNITS[g][j]; line = f"Title: {u['title']}. Passage: {u['text']}"
            key = (g, int(j))
            if key not in _len_cache: _len_cache[key] = LTOK.encode(line, add_special_tokens=False)
            ids = _len_cache[key]
            if used + len(ids) > budget:
                rem = budget - used
                if rem > 0: lines.append(LTOK.decode(ids[:rem]))
                break
            lines.append(line); used += len(ids)
        return lines
    def block(question, lines, answer=None):
        s = INSTR1 + "\n" + "\n".join(lines) + "\n" + INSTR2 + "\nQuestion: " + question + "\nThe answer is"
        return s + (f" {answer}\n\n" if answer is not None else "")
    def closed_block(question, answer=None):
        s = "Answer the following question with just a few words.\nQuestion: " + question + "\nThe answer is"
        return s + (f" {answer}\n\n" if answer is not None else "")

    prompts, meta = [], []
    for d in ["squad", "nq"]:
        demos = [x for x in DEMOS if x["dataset"] == d]; qs = [q for q in QUESTIONS if q["dataset"] == d]
        prefix = "".join(closed_block(x["question"], x["answers"][0]) for x in demos)
        for q in qs:
            prompts.append(prefix + closed_block(q["question"])); meta.append((q["qid"], "closed-book", "-", 0))
        for rname in RETRIEVER_NAMES:
            for g in GRANS:
                I = np.load(f"{WORK}/ranked/{rname}_{g}.npz")["I"]
                for B in EM_BUDGETS:
                    prefix = "".join(block(x["question"], ctx_lines(g, I[QID2I[x["qid"]]], B), x["answers"][0]) for x in demos)
                    for q in qs:
                        prompts.append(prefix + block(q["question"], ctx_lines(g, I[QID2I[q["qid"]]], B)))
                        meta.append((q["qid"], rname, g, B))
    plen = [len(LTOK.encode(p)) for p in prompts[:: max(1, len(prompts)//500)]]
    print(f"{len(prompts)} prompts | prompt tokens (sample) mean {np.mean(plen):.0f}, max {max(plen)}")

    llm = LLM(LLM_NAME, dtype="half", tensor_parallel_size=max(1, torch.cuda.device_count()), max_model_len=4096,
              enable_prefix_caching=True, gpu_memory_utilization=0.90, seed=SEED)
    sp = SamplingParams(temperature=0.0, max_tokens=16, stop=["\n"], seed=SEED)
    t0 = time.perf_counter(); outs = llm.generate(prompts, sp); print(f"generation {time.perf_counter()-t0:.0f}s")

    qa_rows = []
    for (qid, r, g, B), o in zip(meta, outs):
        q = QUESTIONS[QID2I[qid]]; pred = o.outputs[0].text.strip().split("\n")[0].strip().rstrip(".")
        qa_rows.append({"qid": qid, "dataset": q["dataset"], "retriever": r, "granularity": g, "budget": B, "pred": pred,
                        "EM": float(any(normalize_answer(pred) == normalize_answer(a) for a in q["answers"])),
                        "contains": float(has_answer(q["ans_tok"], dpr_tokens(pred)) or
                                          any(normalize_answer(a) in normalize_answer(pred) for a in q["answers"]))})
    pd.DataFrame(qa_rows).to_csv(f"{OUT}/qa_predictions.csv", index=False)
    print("QA xong → Save Version")

# STAGE `analysis`
## 13. Metrics theo query (Recall@k, MRR, gold-page recall, units cần để đủ 20 passage)

In [ ]:
if RUN["analysis"]:
    QID2I = {q["qid"]: i for i, q in enumerate(QUESTIONS + DEMOS)}
    GOLD_OF = {q["qid"]: set(q["gold_pages"]) for q in QUESTIONS}
    RANKED, PER_Q = {}, []
    for rname in RETRIEVER_NAMES:
        for g in GRANS:
            z = np.load(f"{WORK}/ranked/{rname}_{g}.npz"); I = z["I"]; RANKED[(rname, g)] = I
            for q in QUESTIONS:
                row = I[QID2I[q["qid"]]]; seen, top, n_units, u20 = set(), [], 0, np.nan
                for j in row:
                    p = UNIT_PIDX[g][j]; n_units += 1
                    if p not in seen:
                        seen.add(p); top.append(p)
                        if len(top) == 20: u20 = n_units
                        if len(top) == max(K_LIST): break
                hits = [p in q["ans_p"] for p in top]
                first = next((k + 1 for k, h in enumerate(hits) if h), None)
                rec = {"retriever": rname, "granularity": g, "dataset": q["dataset"], "qid": q["qid"],
                       "first_hit_rank": first, "RR": 1.0 / first if first else 0.0, "n_unique_passages": len(top),
                       "units_for_20_passages": u20}
                for k in K_LIST:
                    rec[f"R@{k}"] = float(any(hits[:k]))
                    rec[f"GoldPage@{k}"] = float(any(PAGE_OF_PID[p] in GOLD_OF[q["qid"]] for p in top[:k]))
                PER_Q.append(rec)
    PQ = pd.DataFrame(PER_Q); PQ.to_csv(f"{OUT}/per_query_retrieval.csv", index=False)
    short = PQ[PQ.n_unique_passages < 20].groupby(["retriever", "granularity"]).size()
    if len(short): print("⚠️ <20 passage unique (tăng TOPN_UNITS):", short.to_dict())
    ORDER = pd.MultiIndex.from_product([RETRIEVER_NAMES, GRANS])

## 14. Table 3 — Passage Recall@5 / @20 (+ trend check)

In [ ]:
if RUN["analysis"]:
    def pivot(metric_cols, data=PQ):
        t = data.groupby(["retriever", "granularity", "dataset"])[metric_cols].mean().mul(100).round(1).unstack("dataset")
        t = t.reindex(ORDER); t.columns = [f"{d.upper()} {m}" for m, d in t.columns]
        return t[sorted(t.columns, key=lambda c: (c.split()[0] != "NQ", c))]
    T3 = pivot(["R@5", "R@20"])
    T3["Avg R@5"] = T3[[c for c in T3 if c.endswith("R@5")]].mean(1).round(1)
    T3["Avg R@20"] = T3[[c for c in T3 if c.endswith("R@20")]].mean(1).round(1)
    T3.to_csv(f"{OUT}/table3_recall.csv"); display(T3)

    trend = []
    for (r, d), sub in PQ.groupby(["retriever", "dataset"]):
        for k in [5, 20]:
            m = sub.groupby("granularity")[f"R@{k}"].mean() * 100
            trend.append({"retriever": r, "dataset": d, "k": k, "passage": round(m["passage"], 1), "sentence": round(m["sentence"], 1),
                          "proposition": round(m["proposition"], 1), "Δ prop−pass": round(m["proposition"] - m["passage"], 1),
                          "Prop>Sent>Pass": bool(m["proposition"] > m["sentence"] > m["passage"])})
    TREND = pd.DataFrame(trend); TREND.to_csv(f"{OUT}/trend_check.csv", index=False); display(TREND)

## 15. Recall@k đầy đủ, MRR, gold-page recall

In [ ]:
if RUN["analysis"]:
    import matplotlib.pyplot as plt
    FULL = pivot([f"R@{k}" for k in K_LIST]); FULL.to_csv(f"{OUT}/recall_all_k.csv"); display(FULL)
    MRR = PQ.groupby(["retriever", "granularity", "dataset"])["RR"].mean().mul(100).round(1).unstack("dataset").reindex(ORDER)
    MRR.columns = [f"{c.upper()} MRR@100" for c in MRR.columns]; MRR.to_csv(f"{OUT}/mrr.csv"); display(MRR)
    GP = pivot([f"GoldPage@{k}" for k in [5, 20]]); GP.to_csv(f"{OUT}/gold_page_recall.csv"); display(GP)

    fig, axes = plt.subplots(2, 4, figsize=(18, 7), sharey="row")
    for a, d in enumerate(["nq", "squad"]):
        for b, r in enumerate(RETRIEVER_NAMES):
            ax = axes[a, b]
            for g in GRANS:
                s = PQ[(PQ.dataset == d) & (PQ.retriever == r) & (PQ.granularity == g)]
                ax.plot(K_LIST, [100 * s[f"R@{k}"].mean() for k in K_LIST], marker="o", label=g)
            ax.set_xscale("log"); ax.set_title(f"{r} / {d.upper()}"); ax.set_xlabel("k")
            if b == 0: ax.set_ylabel("Recall@k (%)")
    axes[0, 0].legend(); plt.tight_layout(); plt.savefig(f"{OUT}/recall_at_k_curves.png", dpi=150); plt.show()

## 16. Ý nghĩa thống kê — bootstrap 95% CI + McNemar (paired)
500 câu/dataset nên chênh lệch nhỏ có thể chỉ là nhiễu → kiểm định cặp (cùng câu hỏi, khác granularity).

In [ ]:
if RUN["analysis"]:
    from scipy.stats import binomtest
    B = 2000; rng = np.random.default_rng(SEED)
    ci_rows, sig_rows = [], []
    for (r, d), sub in PQ.groupby(["retriever", "dataset"]):
        piv = {g: sub[sub.granularity == g].set_index("qid") for g in GRANS}
        qids = piv["passage"].index; n = len(qids); idx = rng.integers(0, n, (B, n))
        for k in [5, 20]:
            for g in GRANS:
                h = piv[g].loc[qids, f"R@{k}"].values; boots = h[idx].mean(1) * 100
                ci_rows.append({"retriever": r, "dataset": d, "k": k, "granularity": g, "mean": round(h.mean()*100, 1),
                                "ci_low": round(np.percentile(boots, 2.5), 1), "ci_high": round(np.percentile(boots, 97.5), 1)})
            for g1, g0 in [("proposition", "passage"), ("sentence", "passage"), ("proposition", "sentence")]:
                a = piv[g1].loc[qids, f"R@{k}"].values.astype(bool); b = piv[g0].loc[qids, f"R@{k}"].values.astype(bool)
                win, loss = int((a & ~b).sum()), int((~a & b).sum())
                p = binomtest(win, win + loss, 0.5).pvalue if win + loss else 1.0
                sig_rows.append({"retriever": r, "dataset": d, "k": k, "compare": f"{g1} vs {g0}", "wins": win, "losses": loss,
                                 "ties": int(n - win - loss), "Δ (pp)": round(100 * (a.mean() - b.mean()), 1),
                                 "McNemar p": round(p, 4), "significant (p<0.05)": p < 0.05})
    CI = pd.DataFrame(ci_rows); CI.to_csv(f"{OUT}/bootstrap_ci.csv", index=False)
    SIG = pd.DataFrame(sig_rows); SIG.to_csv(f"{OUT}/mcnemar_win_loss.csv", index=False); display(SIG)

## 17. Recall@l words (Figure 4 / Figure 7) — cả 4 retriever
Ghép các unit retrieve được theo thứ tự, chỉ giữ **l từ đầu** → so sánh công bằng cùng ngân sách độ dài.

In [ ]:
if RUN["analysis"]:
    TEXT_WORDS = {g: [u["text"].split() for u in UNITS[g]] for g in GRANS}
    curves = []
    for (r, g), I in tqdm(RANKED.items(), desc="recall@words"):
        for q in QUESTIONS:
            words = []
            for j in I[QID2I[q["qid"]]]:
                words.extend(TEXT_WORDS[g][j])
                if len(words) >= max(L_WORDS): break
            for l in L_WORDS:
                curves.append({"retriever": r, "granularity": g, "dataset": q["dataset"], "words": l,
                               "hit": has_answer(q["ans_tok"], dpr_tokens(" ".join(words[:l])))})
    CUR = pd.DataFrame(curves).groupby(["retriever", "granularity", "dataset", "words"])["hit"].mean().mul(100).round(1).reset_index()
    CUR.to_csv(f"{OUT}/recall_at_words.csv", index=False)
    fig, axes = plt.subplots(2, 4, figsize=(18, 7), sharey="row")
    for a, d in enumerate(["nq", "squad"]):
        for b, r in enumerate(RETRIEVER_NAMES):
            ax = axes[a, b]
            for g in GRANS:
                s = CUR[(CUR.dataset == d) & (CUR.retriever == r) & (CUR.granularity == g)]; ax.plot(s.words, s.hit, marker="o", label=g)
            ax.set_title(f"{r} / {d.upper()}"); ax.set_xlabel("#Words")
            if b == 0: ax.set_ylabel("Recall (%)")
    axes[0, 0].legend(); plt.tight_layout(); plt.savefig(f"{OUT}/recall_at_words.png", dpi=150); plt.show()

## 18. Recall theo độ "hiếm" của answer (chuyển thể Figure 3)
Paper đo tần suất entity trên EntityQuestions bằng BM25. Ở đây proxy = **số passage trong corpus chứa answer** (ít → thông tin long-tail). Kỳ vọng theo paper: proposition có lợi nhiều hơn ở nhóm hiếm.

In [ ]:
if RUN["analysis"]:
    freq = {q["qid"]: q["ans_freq"] for q in QUESTIONS}
    bins = [0, 1, 3, 10, 30, 10**9]; labels = ["0", "1–2", "3–9", "10–29", "≥30"]
    PQ["ans_freq_bin"] = pd.cut(PQ.qid.map(freq), bins=[-1] + bins[1:], labels=labels, right=False)
    FREQ = (PQ.groupby(["retriever", "dataset", "ans_freq_bin", "granularity"], observed=True)["R@20"].mean().mul(100).round(1)
              .unstack("granularity")[GRANS])
    FREQ["n"] = PQ[PQ.granularity == "passage"].groupby(["retriever", "dataset", "ans_freq_bin"], observed=True).size()
    FREQ.to_csv(f"{OUT}/recall_by_answer_frequency.csv"); display(FREQ.loc["GTR"])
    fig, axes = plt.subplots(1, 4, figsize=(18, 3.8), sharey=True)
    for ax, r in zip(axes, RETRIEVER_NAMES):
        s = PQ[PQ.retriever == r].groupby(["ans_freq_bin", "granularity"], observed=True)["R@20"].mean().mul(100).unstack()[GRANS]
        s.plot(ax=ax, marker="o"); ax.set_title(f"{r} — R@20 vs answer frequency"); ax.set_xlabel("#passages chứa answer")
    plt.tight_layout(); plt.savefig(f"{OUT}/recall_by_answer_frequency.png", dpi=150); plt.show()

## 19. Theo loại câu hỏi (wh-word)

In [ ]:
if RUN["analysis"]:
    def qtype(q):
        w = q.lower().split()
        for t in ["who", "when", "where", "what", "which", "how", "why"]:
            if t in w[:3]: return t
        return "other"
    qt = {q["qid"]: qtype(q["question"]) for q in QUESTIONS}
    PQ["qtype"] = PQ.qid.map(qt)
    QT = PQ.groupby(["dataset", "qtype", "granularity"])["R@20"].mean().mul(100).round(1).unstack("granularity")[GRANS]
    QT["n"] = PQ[(PQ.granularity == "passage")].groupby(["dataset", "qtype"]).size() // len(RETRIEVER_NAMES)
    QT.to_csv(f"{OUT}/recall_by_question_type.csv"); display(QT)

## 20. Hiệu năng (trade-off chi phí index)

In [ ]:
if RUN["analysis"]:
    EFF = pd.read_csv(f"{OUT}/efficiency.csv")
    u20 = PQ.groupby(["retriever", "granularity"])["units_for_20_passages"].mean().round(1).rename("units retrieved for 20 passages")
    EFF = EFF.merge(u20.reset_index(), on=["retriever", "granularity"]); EFF.to_csv(f"{OUT}/efficiency.csv", index=False); display(EFF)

## 21. Error cases (Table 6/7) — top-1 unit của từng granularity

In [ ]:
if RUN["analysis"]:
    cases = []
    for r in ["GTR", "Contriever"]:
        top1 = {g: PQ[(PQ.retriever == r) & (PQ.granularity == g)].set_index("qid")["R@1"] for g in GRANS}
        for q in QUESTIONS:
            h = {g: bool(top1[g][q["qid"]]) for g in GRANS}
            if len(set(h.values())) == 1: continue
            row = {"retriever": r, "dataset": q["dataset"], "question": q["question"], "answers": " | ".join(q["answers"])}
            for g in GRANS:
                u = UNITS[g][RANKED[(r, g)][QID2I[q["qid"]]][0]]
                row[f"{g}_top1"] = f"[{u['title']}] {u['text']}"; row[f"{g}_hit"] = h[g]
            row["pattern"] = "".join("✓" if h[g] else "✗" for g in GRANS)
            cases.append(row)
    CASES = pd.DataFrame(cases); CASES.to_csv(f"{OUT}/error_cases_top1.csv", index=False)
    if CASES.empty:
        print("Không có câu nào mà top-1 của các granularity cho kết quả khác nhau.")
    else:
        print(CASES.groupby(["retriever", "dataset", "pattern"]).size().rename("count"))
        pd.set_option("display.max_colwidth", 200)
        display(CASES[(CASES.retriever == "GTR") & (CASES.pattern == "✗✗✓")].head(3))
        display(CASES[(CASES.retriever == "GTR") & (CASES.pattern == "✓✓✗")].head(3))

## 22. Table 5 — EM@100 / EM@500 (LLaMA-2-7B, nếu đã chạy stage `qa`)

In [ ]:
if RUN["analysis"] and os.path.exists(f"{OUT}/qa_predictions.csv"):
    QA = pd.read_csv(f"{OUT}/qa_predictions.csv")
    cb = QA[QA.retriever == "closed-book"].groupby("dataset")["EM"].mean().mul(100).round(1)
    T5 = QA[QA.retriever != "closed-book"].groupby(["retriever", "granularity", "dataset", "budget"])["EM"].mean().mul(100).round(1)
    T5 = T5.unstack(["dataset", "budget"]).reindex(ORDER); T5.columns = [f"{d.upper()} EM@{b}" for d, b in T5.columns]
    T5 = T5[sorted(T5.columns, key=lambda c: (c.split()[0] != "NQ", int(c.split("@")[1])))]
    T5.loc[("Closed-book", "-"), :] = [cb[c.split()[0].lower()] for c in T5.columns]
    T5.to_csv(f"{OUT}/table5_em.csv"); display(T5)
    CONT = QA[QA.retriever != "closed-book"].groupby(["retriever", "granularity", "dataset", "budget"])["contains"].mean().mul(100).round(1)
    CONT.unstack(["dataset", "budget"]).to_csv(f"{OUT}/qa_contains_answer.csv")

## 22b. Table 4 — EM FiD với top-5 / top-20 passages (nếu đã chạy stage `fid`)

In [ ]:
if RUN["analysis"] and os.path.exists(f"{OUT}/fid_predictions.csv"):
    FIDP = pd.read_csv(f"{OUT}/fid_predictions.csv")
    T4 = FIDP.groupby(["retriever", "granularity", "dataset", "k"])["EM"].mean().mul(100).round(1)
    T4 = T4.unstack(["dataset", "k"]).reindex(ORDER); T4.columns = [f"{d.upper()} EM@top{k}" for d, k in T4.columns]
    T4 = T4[sorted(T4.columns, key=lambda c: (c.split()[0] != "NQ", int(c.split("top")[1])))]
    T4.to_csv(f"{OUT}/table4_fid_em.csv"); display(T4)
    if os.path.exists(f"{OUT}/fid_sanity.json"): print("Sanity:", json.load(open(f"{OUT}/fid_sanity.json")))

    # paired win/loss + McNemar: proposition/sentence vs passage
    from scipy.stats import binomtest
    rows = []
    for (r, d, k), sub in FIDP.groupby(["retriever", "dataset", "k"]):
        piv = sub.pivot(index="qid", columns="granularity", values="EM")
        for g1 in ["proposition", "sentence"]:
            a, b = piv[g1].astype(bool), piv["passage"].astype(bool)
            w, l = int((a & ~b).sum()), int((~a & b).sum())
            p = binomtest(w, w + l, 0.5).pvalue if w + l else 1.0
            rows.append({"retriever": r, "dataset": d, "k": k, "compare": f"{g1} vs passage", "wins": w, "losses": l,
                         "Δ (pp)": round(100 * (a.mean() - b.mean()), 1), "McNemar p": round(p, 4)})
    FID_SIG = pd.DataFrame(rows); FID_SIG.to_csv(f"{OUT}/table4_fid_mcnemar.csv", index=False); display(FID_SIG)

## 23. Tổng hợp → `results/summary.json`

In [ ]:
if RUN["analysis"]:
    SUMMARY = {
        "seed": SEED, "pages": META["pages"], "leak": META["leak"],
        "corpus_stats": CORPUS_STATS.to_dict("records"), "answer_coverage": COVERAGE.reset_index().to_dict("records"),
        "table3": T3.reset_index().to_dict("records"),
        "trend_holds_R@5": TREND[TREND["k"] == 5][["retriever", "dataset", "Prop>Sent>Pass"]].to_dict("records"),
        "significant_prop_vs_pass": SIG[SIG["compare"] == "proposition vs passage"][["retriever", "dataset", "k", "Δ (pp)", "McNemar p"]].to_dict("records"),
    }
    for name, var in [("table4_fid", "T4"), ("table5_llama", "T5")]:
        if var in globals(): SUMMARY[name] = globals()[var].reset_index().astype(str).to_dict("records")
    save_json(SUMMARY, f"{OUT}/summary.json")
    print("Files:", sorted(os.listdir(OUT)))

## 24. Ghi chú cho báo cáo
- **Không leak câu hỏi**: test = SQuAD dev / NQ-open test (giống paper); câu trùng y hệt train đã loại; demo lấy từ train. DPR đã thấy *train split* NQ/SQuAD (paper cũng đánh dấu).
- **Khác paper**: 500 câu/dataset thay vì full test; corpus 8,000 pages (≈0.13% Wikipedia) → Recall tuyệt đối cao hơn; distractor lấy từ các shard chứa gold page; NQ chỉ giữ câu join được title.
- **Lệch phiên bản Wikipedia** (2016/2018 vs 2021): xem `title_mapping_report.csv` và `answer_coverage.csv`.
- Mọi bước ngẫu nhiên dùng `SEED`; decoding greedy → chạy lại cho cùng kết quả (sai khác nhỏ có thể đến từ kernel GPU fp16).
- **FiD (Table 4)**: dùng đúng checkpoint `nq_reader_large` của paper, nạp vào transformers mới (không dùng code FiD gốc do yêu cầu transformers 3.0.2) → có thể lệch nhỏ về số; xem `fid_sanity.json` (EM oracle) để chứng minh nạp đúng. Model train trên NQ nên SQuAD thấp hơn — giống paper (dùng một reader cho mọi dataset).